<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/fine_tuning_Ed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [94]:
# imports

import os
import random
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict

from items import Item

import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import numpy as np
import pickle

In [2]:
!pip install -q typing transformers datasets python_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [6]:
from typing import Optional
from transformers import AutoTokenizer
import re
from datasets import load_dataset

In [62]:
data = load_dataset("pgurazada1/amazon_india_products")["train"]

In [95]:
# Investigate a particular datapoint
datapoint = data[2]

In [96]:
datapoint

{'Uniq Id': '41654633cce38c8650690f6dbac01fd3',
 'Crawl Timestamp': '2019-10-30 09:53:23 +0000',
 'Category': 'Skin Care',
 'Product Title': ' Generic 1 Pc brand snail eye cream remove dark circle eye lifting instant ageless snail cream for eye care anti wrinkle eyes cream 20g ',
 'Product Description': 'Use: eye, item type: cream, net wt: 20g, gzzz: ygzwbz, model number: lkwnys, gender: female, certification: gzzz, feature: moisturizing, dark circle, anti-puffiness, anti-aging, brand name: laikou, certificate number: 2014028630, ingredient: cream, country/region of manufacture: china, eye care features: eye cream, product name: snail extract cream, applicable to the crowd: general, specifications: normal specifications, origin: shantou, guangdong province, unit type: piece, package weight: ,package size:',
 'Brand': 'Generic',
 'Pack Size Or Quantity': None,
 'Mrp': '1824.00',
 'Price': '1042.00',
 'Site Name': 'Amazon In',
 'Offers': '42.87%',
 'Combo Offers': None,
 'Stock Availibil

In [65]:
# How many have prices?

prices = 0
for datapoint in data:
    try:
        price = float(datapoint["Mrp"] or 0.0)
        if price > 0.0:
            prices += 1
    except ValueError as e:
        pass

print(f"There are {prices:,} with prices which is {prices/len(data)*100:,.1f}%")

There are 29,240 with prices which is 97.5%


In [78]:
import pandas as pd
df  = pd.DataFrame(data)

In [79]:
#Drop the rows where the price is none and then delete rows where price is 0.0

df = df.dropna(subset=['Mrp'])



In [80]:
df.dtypes

,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,object
Price,object
Site Name,object


In [84]:
clean_price('2.4.00')

'24.00'

In [85]:
# CHange the data tyoe of the column to float

import re

def clean_price(price_str):
  """Cleans a price string by removing extra decimal points."""
  if price_str.count('.') ==2:
    cleaned_price = re.sub(r"\.", "", price_str.strip(), count=1)
    return cleaned_price
  else:
    return price_str


In [ ]:
# added code to colab

In [86]:
# apply the clean price function

df['Mrp'] = df['Mrp'].apply(lambda x : float(clean_price(x) or 0.0))


In [89]:
df.dtypes

,0
Uniq Id,object
Crawl Timestamp,object
Category,object
Product Title,object
Product Description,object
Brand,object
Pack Size Or Quantity,object
Mrp,float64
Price,object
Site Name,object


Check before we use the Item class if the data frame consists of product description, title and Mrp

In [258]:
#df[df['Product Description'].isna()==True] #1907 rows

#df[df['Product Title'].isna()==True] #0 rows
#df[df['Mrp']==0.0] # 0 rows

,Uniq Id,Crawl Timestamp,Category,Product Title,Product Description,Brand,Pack Size Or Quantity,Mrp,Price,Site Name,Offers,Combo Offers,Stock Availibility,Product Asin,Image Urls


In [261]:
final_df = df[~df['Product Description'].isna()]

In [262]:
final_df.columns

Index(['Uniq Id', 'Crawl Timestamp', 'Category', 'Product Title',
       'Product Description', 'Brand', 'Pack Size Or Quantity', 'Mrp', 'Price',
       'Site Name', 'Offers', 'Combo Offers', 'Stock Availibility',
       'Product Asin', 'Image Urls'],
      dtype='object')

In [267]:

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"

MIN_TOKENS = 150 # Any less than this, and we don't have enough useful content
MAX_TOKENS = 160 # Truncate after this many tokens. Then after adding in prompt text, we will get to around 180 tokens

MIN_CHARS = 300
CEILING_CHARS = MAX_TOKENS * 7

class Item:
    """
    An Item is a cleaned, curated datapoint of a Product with a Price
    """

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    PREFIX = "Price is"
    QUESTION = "How much does this cost to the nearest dollar?"
    REMOVALS = ['"Batteries Included?": "No"', '"Batteries Included?": "Yes"', '"Batteries Required?": "No"', '"Batteries Required?": "Yes"', "By Manufacturer", "Item", "Date First", "Package", ":", "Number of", "Best Sellers", "Number", "Product "]

    title: str
    price: float
    category: str
    token_count: int = 0
    details: Optional[str]
    prompt: Optional[str] = None
    include = False

    def __init__(self, data):
        self.title = data['Product Title']
        self.price = data['Mrp']
        self.parse(data)

    def scrub_details(self):
        """
        Clean up the details string by removing common text that doesn't add value
        """
        details = self.details
        for remove in self.REMOVALS:
            details = details.replace(remove, "")
        return details

    def scrub(self, stuff):
        """
        Clean up the provided text by removing unnecessary characters and whitespace
        Also remove words that are 7+ chars and contain numbers, as these are likely irrelevant product numbers
        """
        stuff = re.sub(r'[\[\]"{}【】\s]+', ' ', stuff).strip()
        stuff = stuff.replace(" ,", ",").replace(",,,",",").replace(",,",",")
        words = stuff.split(' ')
        select = [word for word in words if len(word)<7 or not any(char.isdigit() for char in word)]
        return " ".join(select)

    def parse(self, data):
        """
        Parse this datapoint and if it fits within the allowed Token range,
        then set include to True
        """
        # ADD : Code block to identify and remove the rows where product description is not available
        self.contents = data['Product Description']
        if self.contents:
            self.contents += '\n'
        if len(self.contents) > MIN_CHARS:
            self.contents = self.contents[:CEILING_CHARS]
            text = f"{self.scrub(self.title)}\n{self.scrub(self.contents)}"
            tokens = self.tokenizer.encode(text, add_special_tokens=False)
            if len(tokens) > MIN_TOKENS:
                tokens = tokens[:MAX_TOKENS]
                text = self.tokenizer.decode(tokens)
                self.make_prompt(text)
                self.include = True

    def make_prompt(self, text):
        """
        Set the prompt instance variable to be a prompt appropriate for training
        """
        self.prompt = f"{self.QUESTION}\n\n{text}\n\n"
        self.prompt += f"{self.PREFIX}{str(round(self.price))}.00"
        self.token_count = len(self.tokenizer.encode(self.prompt, add_special_tokens=False))

    def test_prompt(self):
        """
        Return a prompt suitable for testing, with the actual price removed
        """
        return self.prompt.split(self.PREFIX)[0] + self.PREFIX

    def __repr__(self):
        """
        Return a String version of this Item
        """
        return f" {self.QUESTION}\n{self.title}  \n   The description of the product is:   {self.contents} \n {self.PREFIX}= ${self.price}"



In [268]:
def parse(data):
    """
    Parse this datapoint and if it fits within the allowed Token range,
    then set include to True
    """
    contents = data['Product Description']
    return contents

In [269]:
final_df.shape

(27394, 15)

In [271]:
Item(final_df.iloc[27100])

 How much does this cost to the nearest dollar?
 Old Spice Anti-Perspirant 2.6oz Hawkridge Solid (2 Pack)   
   The description of the product is:   Old Spice Anti-Perspirant 2.6oz Hawkridge Solid (2 Pack)
 
 Price is= $4849.0

In [210]:
df.shape[0]

29301

In [272]:
items =[]

for i in range(0, final_df.shape[0]-1):
  try:
    price= float(final_df.iloc[i]["Mrp"])
    if price > 0:
      item = Item(final_df.iloc[i])
      if item.include:
        items.append(item)

  except ValueError as e:
    pass

In [287]:
float(str(items[0]).split('$')[1])

2040.0

In [98]:
#Curate the dataset

items = []
for datapoint in df:
    try:
        price = float(datapoint["Mrp"])
        if price > 0:
            item = Item(datapoint)
            if item.include:
                items.append(item)
    except ValueError as e:
        pass

print(f"There are {len(items):,} items")

TypeError: string indices must be integers

In [ ]:
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B"

In [ ]:
MIN_TOKENS = 150
MAX_TOKENS = 160

MIN_CHARS = 300
CEILING_CHARS = MAX_TOKENS * 7